In [1]:
import pandas as pd
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import torch

In [2]:
df = pd.read_csv('df_cleaned.csv')

In [3]:
df.head(5)

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title and subtitle,tagged_description
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,9780002005883 A NOVEL THAT READERS and critics...
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736 A memorable, mesmerizing heroine..."
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,The Four Loves,9780006280897 Lewis' work on the nature of lov...
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le..."


In [4]:
df['tagged_description'].to_csv('tagged_description.txt', sep='\n', encoding='utf-8', index=False, header = False)

**Load the file and prepare documents for better vector indexing and semantic retrieval.**

In [5]:
raw_doc = TextLoader('tagged_description.txt', encoding='utf-8').load()

In [6]:
# Will store each line as a separate document in a list
documents = []

In [7]:
# Split each line to separate isbn13 and description
for line in raw_doc[0].page_content.splitlines():
    if not line.strip():
        continue
    
    isbn, desc = line.split(" ", 1)

    documents.append(
        Document(
            page_content = desc,
            metadata={"isbn13": isbn}
        )
    )

In [8]:
len(documents)

5197

In [9]:
documents[0]

Document(metadata={'isbn13': '9780002005883'}, page_content='A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a song of celeb

**Setting device on GPU (CUDA) for faster computation.**

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

**Creating Document Embedding. Using "all-MiniLM-L6-v2" model from Hugging Face.**

In [11]:
# Load a local embedding model
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", 
                                        model_kwargs={"device": device}
)

**Applying the model & storing the results in the Chroma vector database.**

In [20]:
db_books = Chroma.from_documents(
    documents,
    embedding=embedding_model,
    persist_directory="chroma_db"
)

In [13]:
# Perform a similarity search
query = "A book about history of India."
docs = db_books.similarity_search(query, k=10)
docs   

[Document(id='2b30d9d5-08d0-43bb-b8fb-e32267cd7dfa', metadata={'isbn13': '9780312151256'}, page_content='A monumental, internationally best-selling novel set in nineteenth-century India weaves a vast tapestry of love, war, and adventure in the foothills of the towering Himalayas. Reprint.'),
 Document(id='fc0ce4cf-a2e7-4da2-b6d4-1492c8a75784', metadata={'isbn13': '9780618074518'}, page_content='The classic novel by the author of Diamond Dust evokes the traumatic history of India after the departure of the British in a story of a Hindu family in Old Delhi and the complex relationships that develop among four people. Reprint.'),
 Document(id='32534b33-d091-488f-8e55-81e247f5232e', metadata={'isbn13': '9780618711666'}, page_content='Presents a novel of life in modern India, chronicling the interwoven journey of an American marine biologist and a Delhi businessman who travel to the remote Sundarban islands.'),
 Document(id='0960a03f-fe7c-48bc-91c4-ba76b0418302', metadata={'isbn13': '978031

**Creating function to retrieve semantic recommendations.**

In [17]:
def retrieve_semantic_recommendations(query: str, top_k: int = 10) -> pd.DataFrame:
    recs = db_books.similarity_search(query, k=50)
    
    # Extract ISBNs directly from metadata
    books_list = [doc.metadata['isbn13'] for doc in recs if 'isbn13' in doc.metadata]
    
    df['isbn13'] = df['isbn13'].astype(str)
    
    # Filter the DataFrame
    return df[df['isbn13'].isin(books_list)][['title', 'authors', 'categories']].head(top_k)

In [19]:
retrieve_semantic_recommendations('a book on friendship')

,title,authors,categories
156,"Men Are from Mars, Women Are from Venus",John Gray,Family & Relationships
215,The Reading Group,Elizabeth Noble,Fiction
254,The Curtain,Milan Kundera,Literary Collections
266,How to Be Popular,Meg Cabot,Juvenile Fiction
365,By the River Piedra I Sat Down and Wept,Paulo Coelho,Fiction
401,Betsy-Tacy,Maud Hart Lovelace,Juvenile Fiction
404,Racso and the Rats of NIMH,Jane Leslie Conly,Juvenile Fiction
527,Dublin 4,Maeve Binchy,Domestic fiction
528,Circle of Friends,Maeve Binchy,College attendance
564,Montaillou,Emmanuel Le Roy Ladurie,Albigenses
